In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

In [3]:
import numpy as np
import pandas as pd

In [5]:
from sentence_transformers import SentenceTransformer

In [6]:
import faiss

## Configuration

In [8]:
RANDOM_STATE = 42

PROCESSED_DIR = Path("../artifacts/processed")
FEATURE_DIR = Path("../artifacts/features")
INDEX_DIR = Path("../artifacts/indexes/faiss")
SEMANTIC_DIR = Path("../artifacts/semantic")

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True,exist_ok=True)
SEMANTIC_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = ("sentence-transformers/all-MiniLM-L6-v2")

TOP_K = 10

RETRIEVAL_K = 100

## Load Data

In [10]:
consumer = pd.read_parquet(PROCESSED_DIR / "consumer_clean.parquet")
content = pd.read_parquet(PROCESSED_DIR / "content_clean.parquet")
print("Consumer shape:", consumer.shape)
print("Content shape:", content.shape)

Consumer shape: (72312, 11)
Content shape: (3122, 16)


## Standardize Columns

In [11]:
consumer.columns = (consumer.columns.str.strip().str.lower())
content.columns = (content.columns.str.strip().str.lower())

## Convert Timestamps

In [12]:
consumer["event_datetime"] = pd.to_datetime(consumer["event_timestamp"],unit="s",utc=True)
content["content_event_datetime"] = pd.to_datetime(content["event_timestamp"],unit="s",utc=True)

## 1. Determine Available Articles

In [14]:
content_sorted = (content.sort_values(["item_id","content_event_datetime"]))

latest_content_state = (content_sorted.groupby("item_id").tail(1).copy())

latest_content_state["is_available"] = (latest_content_state["interaction_type"].astype(str).str.lower().eq("content_present"))

In [15]:
available_items = set(latest_content_state.loc[latest_content_state["is_available"],"item_id"])

pulled_items = set(latest_content_state.loc[~latest_content_state["is_available"],"item_id"])

print("Available:", len(available_items))

print("Pulled:", len(pulled_items))

Available: 2983
Pulled: 74


## 2. English Only Content

In [16]:
semantic_articles = content[content["language"].astype(str).str.lower().eq("en")].copy()

semantic_articles = semantic_articles[semantic_articles["item_id"].isin(available_items)].copy()

semantic_articles = (semantic_articles.drop_duplicates(subset=["item_id"]).reset_index(drop=True))

print("Semantic articles:", len(semantic_articles))

Semantic articles: 2166


## 3. Clean Title And Description

In [17]:
semantic_articles["title"] = (semantic_articles["title"].fillna("").astype(str).str.strip())

semantic_articles["text_description"] = (semantic_articles["text_description"].fillna("").astype(str).str.strip())

## 4. Construct Semantic Text

In [18]:
semantic_articles["article_text"] = ("Title: " + semantic_articles["title"] + ". Description: " +semantic_articles["text_description"])

## 5. Remove Unusable Articles

In [19]:
semantic_articles = semantic_articles[semantic_articles["article_text"].str.replace("Title:", "",regex=False).str.replace("Description:", "",regex=False).str.strip().ne("")].copy()

semantic_articles = (semantic_articles.reset_index(drop=True))

## 6. Load Sentence Transformer

In [20]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model:", EMBEDDING_MODEL_NAME)

Embedding model: sentence-transformers/all-MiniLM-L6-v2


## 7. Generate Article Embeddings

In [21]:
article_texts = (semantic_articles["article_text"].tolist())

article_embeddings = (embedding_model.encode(article_texts, normalize_embeddings=True,batch_size=64, show_progress_bar=True, convert_to_numpy=True))

print("Embedding shape:", article_embeddings.shape)

Batches: 100%|██████████| 34/34 [00:16<00:00,  2.11it/s]

Embedding shape: (2166, 384)


## 8. Verify Embedding Quality

In [22]:
print("Number of articles:", len(article_embeddings))

print("Embedding dimension:", article_embeddings.shape[1])

embedding_norms = np.linalg.norm(article_embeddings, axis=1)

print("Minimum norm:", embedding_norms.min())

print("Maximum norm:", embedding_norms.max())

Number of articles: 2166
Embedding dimension: 384
Minimum norm: 0.9999999
Maximum norm: 1.0000001


## 9. Save Embeddings

In [23]:
embedding_path = (FEATURE_DIR / "article_embeddings.npy")

np.save(embedding_path, article_embeddings)

print("Saved:", embedding_path)

Saved: ..\artifacts\features\article_embeddings.npy


## 10. Save Article Embedding Mapping

In [24]:
embedding_metadata = (semantic_articles[["item_id", "title", "language", "item_type","producer_id"]].copy())

embedding_metadata["embedding_index"] = np.arange(len(embedding_metadata))

embedding_metadata.to_parquet(FEATURE_DIR / "article_embedding_index.parquet", index=False)

print("Saved article embedding metadata.")

Saved article embedding metadata.


## 10. Bulid Item ID Lookup

In [25]:
item_to_embedding_index = {item_id: int(index) for item_id, index in zip(embedding_metadata["item_id"], embedding_metadata["embedding_index"])}

index_to_item = {int(index): item_id for item_id, index in zip(embedding_metadata["item_id"],embedding_metadata["embedding_index"])}

## 11. Bulid FAISS Index

In [28]:
embedding_dimension = (article_embeddings.shape[1])

faiss_index = faiss.IndexFlatIP(embedding_dimension)

faiss_index.add(article_embeddings.astype(np.float32))

print("FAISS index size:", faiss_index.ntotal)

print("Embedding dimension:", faiss_index.d)

FAISS index size: 2166
Embedding dimension: 384


## 12. Save FAISS Index

In [29]:
faiss_index_path = (INDEX_DIR / "article_cosine.index")

faiss.write_index(faiss_index, str(faiss_index_path))

print("Saved:", faiss_index_path)

Saved: ..\artifacts\indexes\faiss\article_cosine.index


## 13. Save Metadata Beside Index

In [30]:
embedding_metadata.to_parquet(INDEX_DIR / "article_metadata.parquet", index=False)

## 14. Search Similar Articles

In [31]:
def search_similar_articles(item_id, top_k=10):
    if item_id not in item_to_embedding_index:
        return pd.DataFrame(
            columns=["item_id", "title", "similarity"])
    embedding_idx = (item_to_embedding_index[item_id])
    query_vector = (article_embeddings[embedding_idx].reshape(1, -1).astype(np.float32))
    scores, indices = (faiss_index.search(query_vector, top_k + 1))
    rows = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        candidate_item = (index_to_item[int(idx)])
        if candidate_item == item_id:
            continue
        rows.append({"item_id": candidate_item, "similarity": float(score)})
    result = pd.DataFrame(rows)
    result = result.merge(embedding_metadata[["item_id", "title", "language", "item_type"]],on="item_id", how="left")
    return result.head(top_k)

## 15. Test Article Similartiy

In [32]:
example_item = (semantic_articles["item_id"].iloc[0])

similar_articles = (search_similar_articles(example_item,top_k=10))

display(similar_articles)

,item_id,similarity,title,language,item_type
0,3353902017498793780,0.739046,The Rise And Growth of Ethereum Gets Mainstrea...,en,HTML
1,5274322067107287523,0.687925,Ethereum and Bitcoin Are Market Leaders But No...,en,HTML
2,8084284001249507595,0.600990,Microsoft Continues to Embrace Ethereum & Bitc...,en,HTML
3,-9171475473795142532,0.587986,Decentralized Options Exchange Etheropt Uses A...,en,HTML
4,5410603488837817052,0.586980,Banks find blockchain hard to put into practic...,en,HTML
5,3067875254349597654,0.526950,Microsoft Adds Ethereum to Windows Platform Fo...,en,HTML
6,607684800821303652,0.511327,How This Former Google Engineer Is Bringing Bl...,en,HTML
7,-4917007328809735647,0.507044,"Google Failure, Ethereum Leaps, ECB Giveout in...",en,HTML
8,8550670510357310628,0.506871,Why Many Smart Contract Use Cases Are Simply I...,en,HTML
9,4849766494522371290,0.505180,Cashila Announces Convenient Buy and Sell Feat...,en,HTML


## 16. User History

In [33]:
consumer_sorted = (consumer.sort_values(["consumer_id", "event_datetime"]).copy())

user_histories = (consumer_sorted.groupby("consumer_id")["item_id"].apply(list).to_dict())

## 17. Interaction Weights

In [34]:
INTERACTION_WEIGHTS = {"content_watched": 1.0, "content_liked": 2.0, "content_saved": 3.0,"content_followed": 4.0, "content_commented_on": 5.0}

consumer["interaction_weight"] = (consumer["interaction_type"].map(INTERACTION_WEIGHTS).fillna(0.0))

## 18. Recency Weighted User Embeddings

In [36]:
RECENCY_HALF_LIFE_DAYS = 14

In [35]:
def calculate_recency_weight(event_time, reference_time):
    age_days = (reference_time - event_time).total_seconds() / 86400
    age_days = max(age_days, 0)
    return (0.5 ** (age_days / RECENCY_HALF_LIFE_DAYS))

## 19. Bulid User Profile Embedding

In [37]:
def build_user_embedding(user_id, reference_time=None):
    user_history = (consumer[consumer["consumer_id"].eq(user_id)].sort_values("event_datetime"))
    if user_history.empty:
        return None
    if reference_time is None:
        reference_time = (consumer["event_datetime"].max())
    weighted_vectors = []
    weights = []
    for _, row in user_history.iterrows():
        item_id = row["item_id"]
        if item_id not in item_to_embedding_index:
            continue
        embedding_idx = (item_to_embedding_index[item_id])
        interaction_weight = (float(row["interaction_weight"]))
        recency_weight = (calculate_recency_weight(row["event_datetime"],reference_time))
        final_weight = (interaction_weight * recency_weight)
        if final_weight <= 0:
            continue
        vector = (article_embeddings[embedding_idx])
        weighted_vectors.append(vector * final_weight)
        weights.append(final_weight)
    if not weighted_vectors:
        return None
    user_vector = (np.sum(weighted_vectors, axis=0) / np.sum(weights))
    norm = np.linalg.norm(user_vector)
    if norm == 0:
        return None
    user_vector = (user_vector / norm)
    return user_vector.astype(np.float32)

## 20. Test User Embedding

In [38]:
example_user = (consumer["consumer_id"].iloc[0])
user_vector = (build_user_embedding(example_user))
if user_vector is not None:
    print("User embedding shape:", user_vector.shape)
    print("User embedding norm:", np.linalg.norm(user_vector))

User embedding shape: (384,)
User embedding norm: 1.0
